# 18. Memory Profiling and Optimization | 显存分析与优化

**难度：** Medium | **环境：** CPU-first | **标签：** `PyTorch`, `显存`, `训练优化` | **目标人群：** Part 2-4 前置补课者

## 本节导读

训练内存不仅包含模型参数，还包含梯度、优化器状态、保存的激活和临时缓冲。这些对象在不同阶段创建、重叠和释放，因此模型能够加载不等于一步训练能够完成。

完成本节后，你应能按对象生命周期估算候选峰值，解释批次、低精度和重计算改变了哪些对象，并用运行时记录识别理论账本的遗漏。

**关键词：** `memory`, `checkpoint`, `accumulation`

![训练显存对象与策略图](../docs/public/00_Prerequisites/18_memory_object_strategy_map.svg)

## 前置阅读
**导语：** 先从 0E 组页判断当前问题是否属于容量证据，再把参数、梯度、激活和临时张量放进同一生命周期账本。
- [0E 组页](./0E.md)


## Q1：训练内存账本为什么必须记录对象和生命周期？

模型参数能够放进 GPU，并不代表训练一定不会发生显存不足（OOM）：前向阶段会保存反向计算所需的激活，反向阶段会产生梯度，优化器状态通常在第一次更新前后完成初始化，算子还可能申请临时缓冲。账本必须同时记录对象大小和可能出现的阶段，才能解释哪些对象有机会在峰值时重叠。


| 显存对象 | 生命周期 | 是否随 batch 变化 | 主要影响因素 |
|:---|:---|:---:|:---|
| 参数 | 模型加载后持续存在 | 否 | 参数量、dtype |
| 梯度 | backward 后产生，更新前保留 | 通常否 | 可训练参数量、梯度 dtype |
| 优化器状态 | 优化器初始化后持续存在 | 否 | 优化器类型、状态精度 |
| 保存的激活 | forward 产生，backward 使用后逐步释放 | 是 | batch、序列长度、层数 |
| 临时缓冲 | 算子执行期间短暂存在 | 是 | 算子实现、workspace、layout |


In [ ]:
import torch
import torch.nn as nn


def pretty_mb(nbytes):
    return f"{nbytes / 1024**2:.2f} MB"


class TinyLedgerNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
        )

    def forward(self, x):
        return self.net(x)


model = TinyLedgerNet()
param_count = sum(p.numel() for p in model.parameters())
param_bytes = param_count * 4
grad_bytes = param_count * 4
adam_state_bytes = param_count * 8

batch_size, seq_len, hidden_size = 16, 64, 512
saved_activation_bytes = batch_size * seq_len * hidden_size * 4
temporary_bytes = saved_activation_bytes // 2

components = {
    'parameters_mb': param_bytes / 1024**2,
    'gradients_mb': grad_bytes / 1024**2,
    'optimizer_state_mb': adam_state_bytes / 1024**2,
    'saved_activations_mb': saved_activation_bytes / 1024**2,
    'temporary_mb': temporary_bytes / 1024**2,
}
phase_members = {
    'after_load': ('parameters_mb',),
    'forward_peak': ('parameters_mb', 'optimizer_state_mb', 'saved_activations_mb', 'temporary_mb'),
    'backward_peak': ('parameters_mb', 'gradients_mb', 'optimizer_state_mb', 'saved_activations_mb', 'temporary_mb'),
    'after_update': ('parameters_mb', 'gradients_mb', 'optimizer_state_mb'),
}
phase_totals = {
    phase: sum(components[name] for name in names)
    for phase, names in phase_members.items()
}
ledger = {
    'objects_mb': components,
    'phase_members': phase_members,
    'phase_totals_mb': phase_totals,
    'overlap_upper_bound_mb': max(phase_totals.values()),
}
print('objects:', {name: round(value, 2) for name, value in components.items()})
print('phase totals:', {name: round(value, 2) for name, value in phase_totals.items()})


## Q1 验证：对象是否只计入可能出现的阶段？

加载后只应存在参数；反向峰值候选应同时包含梯度、优化器状态、保存的激活和临时缓冲。


In [ ]:
assert phase_members['after_load'] == ('parameters_mb',)
assert {'gradients_mb', 'saved_activations_mb', 'temporary_mb'} <= set(phase_members['backward_peak'])
assert ledger['overlap_upper_bound_mb'] == phase_totals['backward_peak']
assert all(value >= 0 for value in components.values())
print('✅ 对象生命周期与候选峰值阶段通过')


## Q2：为什么对象大小不能直接相加成真实峰值？

真实峰值取决于对象是否同时存活。加载阶段主要保留参数；前向峰值加入保存的激活和临时缓冲；反向阶段还可能同时存在梯度。理论账本可以给出候选峰值阶段和对象贡献，但 allocator、算子 workspace 和释放时机仍可能让实测结果偏离理论值。

| 阶段 | 新进入或仍存活的主要对象 | 账本能说明什么 |
|:---|:---|:---|
| 加载后 | 参数 | 模型状态的最低驻留要求 |
| 前向峰值候选 | 参数、优化器状态、保存的激活、临时缓冲 | batch 与序列长度带来的动态压力 |
| 反向峰值候选 | 前向对象、梯度 | 多类状态可能重叠的理论上界 |
| 更新后 | 参数、梯度、优化器状态 | 更新阶段结束前的长期状态 |


In [ ]:
def summarize_candidate_peak(objects_mb, phases):
    """按生命周期计算候选峰值阶段和该阶段的对象贡献。"""
    if any(value < 0 for value in objects_mb.values()):
        raise ValueError('对象大小不能为负数')
    totals = {phase: sum(objects_mb[name] for name in names) for phase, names in phases.items()}
    peak_phase = max(totals, key=totals.get)
    contributions = {name: objects_mb[name] for name in phases[peak_phase]}
    return {
        'peak_phase': peak_phase,
        'candidate_peak_mb': totals[peak_phase],
        'phase_totals_mb': totals,
        'contributions_mb': contributions,
    }


peak_report = summarize_candidate_peak(components, phase_members)
print('candidate peak:', peak_report)


## Q2 验证：候选峰值是否由生命周期重叠决定？

候选峰值必须来自某个明确阶段，并且阶段总量等于该阶段存活对象之和。


In [ ]:
assert peak_report['peak_phase'] == 'backward_peak'
assert peak_report['candidate_peak_mb'] == sum(peak_report['contributions_mb'].values())
assert peak_report['phase_totals_mb']['after_load'] < peak_report['candidate_peak_mb']
print('✅ 生命周期重叠与候选峰值计算通过')


## Q3：什么时候该缩 batch，什么时候该做梯度累积？

先区分两个 batch：`micro-batch` 是一次真正送入模型的样本数，`effective batch` 是累积若干次梯度后才更新一次参数的总样本数。缩小 micro-batch 会降低单次 activation 峰值；梯度累积可以近似保住 effective batch，但会增加前向 / 反向次数，不会减少参数、梯度或 optimizer state 的长期占用。

| 策略 | 主要减少什么 | 主要代价 |
|:---|:---|:---|
| 缩小 micro-batch | 单次 activation 峰值 | 单次处理量变小 |
| 增加 gradient accumulation | 在较小 micro-batch 下保持 effective batch | 前向 / 反向次数增加 |
| 两者配合 | 在容量限制下维持目标更新规模 | 吞吐和训练时间需要重新测量 |


In [ ]:
configs = [
    {"name": "baseline", "micro_batch": 16, "accum_steps": 1, "world_size": 1},
    {"name": "accumulated", "micro_batch": 4, "accum_steps": 4, "world_size": 1},
]

seq_len, hidden_size, dtype_bytes = 64, 512, 2

def mb(nbytes):
    """把字节数转换为便于阅读的 MB。"""
    return nbytes / 1024**2

for cfg in configs:
    effective_batch = cfg["micro_batch"] * cfg["accum_steps"] * cfg["world_size"]
    activation_bytes = cfg["micro_batch"] * seq_len * hidden_size * dtype_bytes
    forward_backward_rounds = cfg["accum_steps"]
    cfg['effective_batch'] = effective_batch
    cfg['activation_peak_mb'] = mb(activation_bytes)
    cfg['forward_backward_rounds'] = forward_backward_rounds
    print(cfg)


## Q3 验证：相同 effective batch 下，micro-batch 与累计轮数是否形成交换？

缩小 micro-batch 应降低 activation 峰值；为了保持相同 effective batch，需要增加 forward/backward 轮数。


In [ ]:
assert configs[0]['effective_batch'] == configs[1]['effective_batch']
assert configs[1]['activation_peak_mb'] < configs[0]['activation_peak_mb']
assert configs[1]['forward_backward_rounds'] > configs[0]['forward_backward_rounds']
print('✅ effective batch、activation 峰值和累计轮数关系通过')


## Q4：混合精度和梯度检查点分别省什么？

如果账本显示保存的激活占比较高，可以减少每个元素的字节数，或减少反向前需要保留的中间结果。低精度主要改变元素字节数；梯度检查点改变保存比例，并在反向阶段重算部分前向。下面显式给出“保存比例”和“额外重算比例”，只比较方向，不把它们当成任意模型的固定收益。


In [ ]:
batch_size, seq_len, hidden_size = 8, 32, 512

def activation_tradeoff(dtype_bytes, saved_fraction, extra_forward_fraction):
    """在给定保存比例假设下比较激活字节数与重算代价。"""
    if dtype_bytes <= 0 or not 0 < saved_fraction <= 1:
        raise ValueError('dtype_bytes 必须为正，saved_fraction 必须位于 (0, 1]')
    if not 0 <= extra_forward_fraction <= 1:
        raise ValueError('extra_forward_fraction 必须位于 [0, 1]')
    full_bytes = batch_size * seq_len * hidden_size * dtype_bytes
    return {
        'saved_activation_mb': full_bytes * saved_fraction / 1024**2,
        'extra_forward_fraction': extra_forward_fraction,
    }

rows = [
    ('fp32 / no checkpoint', activation_tradeoff(4, 1.0, 0.0)),
    ('bf16 / no checkpoint', activation_tradeoff(2, 1.0, 0.0)),
    ('fp32 / save 50%', activation_tradeoff(4, 0.5, 0.5)),
    ('bf16 / save 50%', activation_tradeoff(2, 0.5, 0.5)),
]

for name, result in rows:
    print(name, '->', result)


## Q4 验证：dtype 与 checkpoint 是否分别改变字节数和重算代价？

低精度应减少同一保存比例下的字节数；减少保存比例应降低保存的激活，同时增加额外前向计算。


In [ ]:
assert rows[1][1]['saved_activation_mb'] < rows[0][1]['saved_activation_mb']
assert rows[2][1]['saved_activation_mb'] < rows[0][1]['saved_activation_mb']
assert rows[3][1]['saved_activation_mb'] < rows[1][1]['saved_activation_mb']
assert rows[2][1]['extra_forward_fraction'] > rows[0][1]['extra_forward_fraction']
print('✅ 元素字节数、保存比例与重算代价关系通过')


## Q5：为什么内存会随训练轮次持续增长？

如果占用不是在某一步突然升高，而是每轮训练后都继续增长，先检查是否把带计算图的 Tensor 或损失持续放进列表。这是 Python 引用延长了计算图生命周期，不应笼统归为 allocator 泄漏。`loss.item()` 只保存数字，`detach()` 保留 Tensor 但切断当前计算图，`no_grad()` 则关闭一段代码的梯度追踪；三者解决的不是同一个问题。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


torch.manual_seed(0)
model = nn.Linear(4, 2)
x = torch.randn(8, 4)
y = torch.randint(0, 2, (8,))

loss_bucket = []
for _ in range(2):
    out = model(x)
    loss = F.cross_entropy(out, y)
    loss_bucket.append(loss)

scalar_bucket = []
for _ in range(2):
    out = model(x)
    loss = F.cross_entropy(out, y)
    scalar_bucket.append(loss.item())

with torch.no_grad():
    pred = model(x)

detached = model(x).detach()

print(f"keep tensor: type={type(loss_bucket[0]).__name__}, grad_fn={loss_bucket[0].grad_fn is not None}")
print(f"keep scalar: type={type(scalar_bucket[0]).__name__}, value={scalar_bucket[0]:.4f}")
print(f"no_grad output requires_grad={pred.requires_grad}")
print(f"detach output requires_grad={detached.requires_grad}, grad_fn={detached.grad_fn}")
print("store tensors only when you really need the graph")


## Q5 验证：保存 Tensor、标量和切断计算图是否产生不同状态？

直接保存 loss Tensor 会保留 `grad_fn`；保存标量、使用 `no_grad()` 或 `detach()` 不应继续持有当前计算图。


In [ ]:
assert loss_bucket[0].grad_fn is not None
assert isinstance(scalar_bucket[0], float)
assert not pred.requires_grad
assert not detached.requires_grad and detached.grad_fn is None
print('✅ 计算图持有、标量保存、no_grad 和 detach 区分通过')


## Q6：如何把理论账本与 `allocated`、`reserved` 和 `peak` 对照？

理论账本描述已知对象及其候选重叠，运行时记录描述分配器看到的实际占用。`allocated_peak` 明显高于理论上界时，应检查遗漏的 workspace 或状态；`reserved_peak` 明显高于 `allocated_peak` 时，应检查缓存和碎片；两者都接近设备容量时，才说明当前 workload 的容量余量不足。

| 对照量 | 计算方式 | 下一项检查 |
|:---|:---|:---|
| 未建模占用 | `allocated_peak - theoretical_upper_bound` | workspace、框架状态或遗漏对象 |
| 分配器保留差值 | `reserved_peak - allocated_peak` | 缓存、碎片和分配策略 |
| 容量余量 | `capacity - reserved_peak` | 是否需要缩小当前 workload |


In [ ]:
def reconcile_memory_record(theoretical_upper_bound_mb, allocated_peak_mb, reserved_peak_mb, capacity_mb):
    """对照理论候选上界和运行时记录，返回仍需检查的证据。"""
    values = (theoretical_upper_bound_mb, allocated_peak_mb, reserved_peak_mb, capacity_mb)
    if min(values) < 0 or capacity_mb <= 0:
        raise ValueError('内存记录必须非负，capacity_mb 必须大于 0')
    if reserved_peak_mb < allocated_peak_mb:
        raise ValueError('reserved_peak_mb 不应小于 allocated_peak_mb')
    unmodeled_mb = max(0.0, allocated_peak_mb - theoretical_upper_bound_mb)
    allocator_gap_mb = reserved_peak_mb - allocated_peak_mb
    headroom_mb = capacity_mb - reserved_peak_mb
    if unmodeled_mb > theoretical_upper_bound_mb * 0.10:
        next_check = 'inspect_unmodeled_objects'
    elif allocator_gap_mb > capacity_mb * 0.10:
        next_check = 'inspect_allocator_cache_or_fragmentation'
    elif headroom_mb < capacity_mb * 0.10:
        next_check = 'capacity_pressure'
    else:
        next_check = 'record_consistent'
    return {
        'unmodeled_mb': round(unmodeled_mb, 1),
        'allocator_gap_mb': round(allocator_gap_mb, 1),
        'headroom_mb': round(headroom_mb, 1),
        'next_check': next_check,
    }


runtime_record = reconcile_memory_record(
    theoretical_upper_bound_mb=9000,
    allocated_peak_mb=9300,
    reserved_peak_mb=11000,
    capacity_mb=12000,
)
print('runtime record:', runtime_record)


## Q6 验证：三类差值是否指向正确的下一项检查？

示例中的分配器保留差值超过容量的 10%，应先检查缓存或碎片，而不是直接断言某个优化策略有效。


In [ ]:
assert runtime_record['unmodeled_mb'] == 300
assert runtime_record['allocator_gap_mb'] == 1700
assert runtime_record['headroom_mb'] == 1000
assert runtime_record['next_check'] == 'inspect_allocator_cache_or_fragmentation'
print('✅ 理论账本与运行时内存记录对账通过')


## 相关阅读
**导语：** 完成本节后，可以查阅真实分配器字段，再把容量证据带入配对实验或训练策略复测。
- [PyTorch CUDA 显存管理文档](https://docs.pytorch.org/docs/stable/notes/cuda.html)
- [Part 00 · 20 性能证据与实验设计](./20_Performance_Evidence_and_Experiment_Design.ipynb)
- [Part 02 · 76 激活检查点与卸载对比](../02_PyTorch_Algorithms/76_Activation_Checkpoint_Offload_Benchmark.ipynb)
